## Using Keras to Build and Train Neural Networks

In this assignment you will use a neural network to predict diabetes using the Pima Diabetes Dataset.  You will use the Keras package to quickly build and train a neural network and compare the performance. 


## UCI Pima Diabetes Dataset

* UCI ML Repositiory (http://archive.ics.uci.edu/ml/datasets/Pima+Indians+Diabetes)

### Attributes: (all numeric-valued)
   1. Number of times pregnant
   2. Plasma glucose concentration a 2 hours in an oral glucose tolerance test
   3. Diastolic blood pressure (mm Hg)
   4. Triceps skin fold thickness (mm)
   5. 2-Hour serum insulin (mu U/ml)
   6. Body mass index (weight in kg/(height in m)^2)
   7. Diabetes pedigree function
   8. Age (years)
   9. Class variable (0 or 1)

The UCI Pima Diabetes Dataset which has 8 numerical predictors and a binary outcome.

## Questions

### Part 1: Data Exploration and Preprocessing

1. Read and load data into Python
2. Explore and pre-process the dataset. For examples;
    1. Handle Missing values
    2. Check Duplicate values 
    3. Outliers detection
    4. Check correlation
    5. Check imbalanced data
    6. Scale or Normalize data
    6. Plots: Histograms, Boxplots, pairplot, etc. 
  
  
### Part 2: Build a Baseline Model

Use the Sequential model to quickly build a baseline neural network with one single hidden layer with 12 nodes. 

3. Split the data to training and testing dataset (75%, 25%)
4. Build the baseline model and find how many parameters does your model have?
5. Train you model with 20 epochs with RMSProp at a learning rate of .001 and a batch size of 128
6. Graph the trajectory of the loss functions, accuracy on both train and test set.
7. Evaluate and interpret the accuracy and loss performance during training, and testing. 

### Part 3: Find the Best Model

Now  try four different models and see if you can improve the accuracy by focusing on different network structures (i.e, activation functions, optimization algorithms, batch sizes, number of epochs, ...), affecting the performance, training time, and level of overfitting (or underfitting).

8. For all your models, plot the ROC curve for the predictions.
9. Which model has best performance, why?
10. Save your best model weights into a binary file.


Submit two files: the Jupyter notebook with your code and answers and its print out PDF.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# ----------------------------
# Part 1: Data Exploration and Preprocessing
# ----------------------------

csv_path = Path('pima-indians-diabetes.csv')
columns = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]

# 1) Read and load the data
raw_df = pd.read_csv(csv_path, header=None, names=columns).astype(float)
df = raw_df.copy()

print('1. Dataset shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())
print('\nData types:')
print(df.dtypes)
print('\nSummary statistics:')
print(df.describe())

# 2) Handle missing values
print('\n2. Missing values in the original data:')
print(df.isnull().sum())

# Zero values for medical measurements are not valid, so treat them as missing.
zero_as_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_clean = df.copy()
df_clean.loc[:, zero_as_missing] = df_clean.loc[:, zero_as_missing].replace(0, np.nan)

print('\nMissing values after replacing 0 with NaN:')
print(df_clean.isnull().sum())

# 3) Check duplicate values
duplicates = df_clean.duplicated().sum()
print('\n3. Duplicate rows:', duplicates)

# 4) Check class imbalance
print('\n4. Class distribution (Outcome):')
print(df_clean['Outcome'].value_counts().sort_index())

plt.figure(figsize=(6, 4))
sns.countplot(data=df_clean, x='Outcome')
plt.title('Class distribution (Outcome)')
plt.xlabel('Outcome')
plt.ylabel('Count')
plt.show()

# 5) Correlation analysis
print('\n5. Correlation matrix:')
correlation_matrix = df_clean.corr(numeric_only=True)
print(correlation_matrix.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

# 6) Outlier detection using IQR
print('\n6. Outlier counts using IQR rule:')
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.drop('Outcome')
for col in numeric_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df_clean[(df_clean[col] < lower) | (df_clean[col] > upper)]
    print(f'  {col}: {len(outliers)} outliers (lower={lower:.2f}, upper={upper:.2f})')

# 7) Histograms and boxplots
print('\n7. Histograms and boxplots for numeric features:')
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.histplot(df_clean[col], bins=20, kde=True, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.boxplot(y=df_clean[col], ax=axes[i])
    axes[i].set_title(f'Boxplot of {col}')
    axes[i].set_ylabel(col)
plt.tight_layout()
plt.show()

# 8) Scale / normalize the data
print('\n8. Scaling the feature matrix with StandardScaler:')
median_values = df_clean[zero_as_missing].median()
df_clean[zero_as_missing] = df_clean[zero_as_missing].fillna(median_values)

scaler = StandardScaler()
X = df_clean.drop(columns=['Outcome'])
y = df_clean['Outcome']
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled_df['Outcome'] = y.values

print('Scaled feature preview:')
print(X_scaled_df.head())
print('\nFinal cleaned data shape:', df_clean.shape)
print('Scaled feature matrix shape:', X_scaled.shape)
